# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 | 
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [16]:
!pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")

Setup complete!


In [17]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence â‰¥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [18]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

#  1c. Filter LLM labels by confidence
llm_X = vec.transform(llm["review"])
llm_probs = clf.predict_proba(llm_X)
llm_confidence = llm_probs.max(axis=1)
llm_predictions = clf.predict(llm_X)

filtered_llm = llm[
    (llm_confidence >= 0.65) & (llm_predictions == llm["label"])
].copy()
print(f"Filtered LLM samples (conf >= 0.65 & agree): {len(filtered_llm)}")

#  1d. Merge & deduplicate
consolidated = pd.concat([gold, weak, filtered_llm], ignore_index=True)
consolidated = consolidated.drop_duplicates(subset="review").reset_index(drop=True)
consolidated.to_csv("consolidated_base.csv", index=False)
print(f"consolidated_base.csv saved: {len(consolidated)} unique reviews")

#  1e. Class distribution analysis
class_counts = consolidated["label"].value_counts()
minority_class = class_counts.idxmin()
print(f"Class distribution:\n{class_counts}")
print(f"Minority class: {minority_class}")

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(class_counts.index, class_counts.values,
              color=["#4CAF50", "#F44336", "#2196F3"][:len(class_counts)])
ax.set_title("Class Distribution of Consolidated Dataset", fontsize=14)
ax.set_xlabel("Sentiment")
ax.set_ylabel("Count")
for bar, val in zip(bars, class_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=150)
plt.close()
print("class_distribution.png saved.")

Gold: 100, Weak: 220, LLM: 150
Filtered LLM samples (conf >= 0.65 & agree): 27
consolidated_base.csv saved: 328 unique reviews
Class distribution:
label
Negative    151
Neutral     115
Positive     62
Name: count, dtype: int64
Minority class: Positive
class_distribution.png saved.


In [19]:

#  1f. Augmentation functions

def synonym_replacement(text, replace_frac=0.15):
    """Replace 15-20% of words with WordNet synonyms. Preserve sentiment-bearing words."""
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)

    # Map Penn POS tags to WordNet POS tags
    def get_wn_pos(tag):
        if tag.startswith("J"):
            return wordnet.ADJ
        elif tag.startswith("V"):
            return wordnet.VERB
        elif tag.startswith("N"):
            return wordnet.NOUN
        elif tag.startswith("R"):
            return wordnet.ADV
        return None

    # Identify replaceable indices: not preserve words, not punctuation
    replaceable = [
        i for i, (word, tag) in enumerate(tagged)
        if word.lower() not in PRESERVE_WORDS
        and word.isalpha()
        and get_wn_pos(tag) is not None
    ]

    n_replace = max(1, int(len(replaceable) * replace_frac))
    indices_to_replace = random.sample(replaceable, min(n_replace, len(replaceable)))

    new_tokens = tokens[:]
    for i in indices_to_replace:
        word, tag = tagged[i]
        wn_pos = get_wn_pos(tag)
        synsets = wordnet.synsets(word, pos=wn_pos)
        synonyms = []
        for syn in synsets:
            for lemma in syn.lemmas():
                candidate = lemma.name().replace("_", " ")
                if candidate.lower() != word.lower() and candidate.isalpha():
                    synonyms.append(candidate)
        if synonyms:
            new_tokens[i] = random.choice(synonyms)

    return " ".join(new_tokens)


def back_translate(text, src="en", mid="hi"):
    """Translate English â†’ Hindi â†’ English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    try:
        hindi = GoogleTranslator(source=src, target=mid).translate(text)
        time.sleep(0.5)  # rate-limit sleep
        back = GoogleTranslator(source=mid, target=src).translate(hindi)
        time.sleep(0.5)
        return back if back else text
    except Exception as e:
        print(f"Back-translation failed: {e}")
        return text


def quality_filter(original, augmented):
    """Return True if augmented text passes Jaccard similarity (0.30â€“0.95)."""
    set_orig = set(original.lower().split())
    set_aug  = set(augmented.lower().split())
    union = set_orig | set_aug
    if not union:
        return False
    jaccard = len(set_orig & set_aug) / len(union)
    return 0.30 <= jaccard <= 0.95


#  1g. Apply augmentation to minority class
minority_samples = consolidated[consolidated["label"] == minority_class].copy()
print(f"Minority class '{minority_class}' has {len(minority_samples)} samples. Augmenting...")

augmented_rows = []
for _, row in minority_samples.iterrows():
    original_text = row["review"]
    label = row["label"]

    # Method 1: Synonym Replacement
    syn_aug = synonym_replacement(original_text, replace_frac=random.uniform(0.15, 0.20))
    if syn_aug and quality_filter(original_text, syn_aug):
        augmented_rows.append({"review": syn_aug, "label": label, "augmentation": "synonym_replacement"})

    # Method 2: Back Translation
    bt_aug = back_translate(original_text)
    if bt_aug and quality_filter(original_text, bt_aug):
        augmented_rows.append({"review": bt_aug, "label": label, "augmentation": "back_translation"})

augmented_df = pd.DataFrame(augmented_rows)
print(f"Augmented samples generated (after quality filter): {len(augmented_df)}")

# Combine original consolidated with augmented minority samples
augmented_classical = pd.concat(
    [consolidated.assign(augmentation="original"), augmented_df],
    ignore_index=True
)
augmented_classical = augmented_classical.drop_duplicates(subset="review").reset_index(drop=True)
augmented_classical.to_csv("augmented_classical.csv", index=False)
print(f"augmented_classical.csv saved: {len(augmented_classical)} total rows")

Minority class 'Positive' has 62 samples. Augmenting...
Augmented samples generated (after quality filter): 123
augmented_classical.csv saved: 449 total rows


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [20]:
from openai import OpenAI
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

g
USE_API = bool(OPENROUTER_API_KEY)

if USE_API:
    try:
        client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)
        MODEL = "mistral/mistral-7b-instruct:free"  # Free model on OpenRouter
        print("✓ Using OpenRouter API with Mistral-7B (free)")
    except Exception as e:
        print(f"⚠ OpenRouter initialization failed: {e}")
        USE_API = False
else:
    print("⚠ OPENROUTER_API_KEY not set. Using local synthetic review generator.")
    USE_API = False

BATCH_SIZE = 20
TARGET = {"Positive": 150, "Negative": 100, "Neutral": 50}
SELF_BLEU_THRESHOLD = 0.7

#  2a. Design your few-shot prompt 
# Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]


gold_df = pd.read_csv("gold_standard_100.csv")

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_gold = tfidf.fit_transform(gold_df["review"])
y_gold = gold_df["label"]

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_gold, y_gold)

def get_few_shot_examples(df, n_per_class=1):
    examples = []
    for label in ["Positive", "Negative", "Neutral"]:
        subset = df[df["label"] == label]
        sample = subset.sample(min(n_per_class, len(subset)), random_state=42)
        for _, row in sample.iterrows():
            examples.append({
                "review": row["review"],
                "sentiment": label
            })
    return examples

few_shot_examples = get_few_shot_examples(gold_df, n_per_class=1)

FEW_SHOT_BLOCK = "\n".join(
    [f'  {{"review": {json.dumps(e["review"])}, "sentiment": "{e["sentiment"]}", "movie": "Example Movie"}}'
     for e in few_shot_examples]
)

PROMPT_TEMPLATE = """You are a movie review generator.

Generate exactly {batch_size} realistic movie reviews with sentiment: {sentiment}.

Rules:
- Ensure sentiment is clearly {sentiment}
- Vary length and style
- Use different movie names
- Avoid repetition
- Output ONLY JSON array

Format:
[
  {{"review": "...", "sentiment": "{sentiment}", "movie": "..."}}
]

Examples:
[
{examples}
]

Now generate {batch_size} reviews in JSON format only:
"""

# Save prompt
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

print("Few-shot prompt designed and saved to prompt_template.txt\n")

#  2b. Generate reviews in batches 
# Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors

def call_openrouter(sentiment, batch_size, retries=2):
    """Call OpenRouter API to generate reviews with error handling."""
    if not USE_API:
        return None  
    
    prompt = PROMPT_TEMPLATE.format(
        batch_size=batch_size,
        sentiment=sentiment,
        examples=FEW_SHOT_BLOCK
    )

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8,
                max_tokens=2000
            )

            raw = response.choices[0].message.content.strip()

           
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
                raw = raw.strip()

            parsed = json.loads(raw)

            results = []
            for item in parsed:
                results.append({
                    "review": str(item.get("review", "")).strip(),
                    "label": sentiment,
                    "movie": str(item.get("movie", "Unknown")).strip()
                })

            return results

        except json.JSONDecodeError:
            time.sleep(1)
        except Exception as e:
            time.sleep(1)

    return None


def generate_local_reviews(sentiment, batch_size):
    """Generate synthetic reviews locally without API."""
    templates = {
        "Positive": [
            "I absolutely loved this movie! {movie} is a masterpiece.",
            "One of the best films I've seen! {movie} was incredible and moving.",
            "Amazing! {movie} captured my heart completely.",
            "Fantastic film! {movie} exceeded all my expectations.",
            "Brilliant storytelling in {movie}. Highly recommended!",
            "Simply wonderful! {movie} is a must-watch.",
            "Exceptional screenplay and acting in {movie}!",
            "Outstanding performance! {movie} is definitely worth watching.",
            "Loved every second of {movie}. Perfect entertainment!",
            "Beautiful and engaging - {movie} is a gem!"
        ],
        "Negative": [
            "Terrible movie. {movie} was a waste of my time.",
            "I disliked {movie} very much. Poor plot and acting.",
            "Boring! {movie} had no redeeming qualities.",
            "Awful film. {movie} was hard to sit through.",
            "Disappointing and dull. {movie} missed the mark.",
            "Bad acting and weak plot in {movie}.",
            "Unwatchable! {movie} was poorly executed.",
            "Frustrating mess. {movie} failed on every level.",
            "Pathetic screenplay. {movie} was dreadful.",
            "Horrible production. {movie} is not recommended."
        ],
        "Neutral": [
            "{movie} was okay, nothing special.",
            "It was a decent movie. {movie} had its moments.",
            "Average film. {movie} was neither good nor bad.",
            "{movie} is watchable, but forgettable.",
            "Mediocre production. {movie} was somewhat enjoyable.",
            "Neither impressed nor disappointed by {movie}.",
            "{movie} has some strengths and weaknesses.",
            "Typical filmmaking. {movie} is standard entertainment.",
            "Fair movie. {movie} is reasonable for a lazy evening.",
            "{movie} was moderately entertaining."
        ]
    }
    
    movies = [
        "The Cosmic Journey", "Silent Shadows", "Urban Dreams", "Ocean's Mystery",
        "Desert Rose", "Mountain Peak", "Endless Night", "Crystal Palace",
        "Broken Promises", "Starlight Echo", "Thunder Storm", "Midnight Rain",
        "Sunset Boulevard", "Morning Glory", "Eternal Fire", "Frozen Heart"
    ]
    
    results = []
    template_list = templates.get(sentiment, templates["Neutral"])
    
    for i in range(batch_size):
        template = random.choice(template_list)
        movie = random.choice(movies)
        review = template.format(movie=movie)
        
        results.append({
            "review": review,
            "label": sentiment,
            "movie": movie
        })
    
    return results


print("2b. Generating reviews in batches...")
all_generated = []

for sentiment, total in TARGET.items():
    print(f"\nGenerating {total} {sentiment} reviews:")
    collected = []

    batches = [BATCH_SIZE] * (total // BATCH_SIZE)
    if total % BATCH_SIZE:
        batches.append(total % BATCH_SIZE)

    for batch_idx, b in enumerate(batches, 1):
        print(f"  Batch {batch_idx}/{len(batches)} ({b} reviews)...", end=" ")
        
        # Try API first
        results = call_openrouter(sentiment, b)
        
        # Fallback to local generator if API fails
        if results is None:
            if USE_API and batch_idx == 1:
                print("API failed, using local generator")
            results = generate_local_reviews(sentiment, b)
        
        collected.extend(results)
        print(f"✓ Got {len(results)} reviews")
        time.sleep(0.5)

    all_generated.extend(collected[:total])
    print(f"  Total {sentiment} collected: {len(collected)}")


print(f"\nProcessing generated reviews...")
gen_df = pd.DataFrame(all_generated)
print(f"  Initial count: {len(gen_df)}")

gen_df = gen_df.drop_duplicates(subset="review")
print(f"  After deduplication: {len(gen_df)}")

gen_df = gen_df[gen_df["review"].str.len() > 10]
print(f"  After filtering (length > 10): {len(gen_df)}")

#  2c. Diversity metrics 
# Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score

def compute_self_bleu(texts, smoothing_function=None):
    """Compute average Self-BLEU score for a list of texts."""
    if len(texts) < 2:
        return 0.0
    
    if smoothing_function is None:
        smoothing_function = SmoothingFunction().method1
    
    bleu_scores = []
    
    for i, hypothesis in enumerate(texts):
        references = [texts[j] for j in range(len(texts)) if j != i]
        
        hyp_tokens = hypothesis.lower().split()
        ref_tokens_list = [ref.lower().split() for ref in references]
        
        max_bleu = 0.0
        for ref_tokens in ref_tokens_list:
            try:
                bleu = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5, 0.5), 
                                    smoothing_function=smoothing_function)
                max_bleu = max(max_bleu, bleu)
            except:
                pass
        
        bleu_scores.append(max_bleu)
    
    return np.mean(bleu_scores) if bleu_scores else 0.0


print("\n2c. Computing Self-BLEU diversity metrics...")
diversity_stats = {}
for sentiment in TARGET.keys():
    sentiment_reviews = gen_df[gen_df["label"] == sentiment]["review"].tolist()
    self_bleu = compute_self_bleu(sentiment_reviews)
    diversity_stats[sentiment] = {
        "count": len(sentiment_reviews),
        "self_bleu": self_bleu,
        "quality": "HIGH" if self_bleu >= SELF_BLEU_THRESHOLD else "LOW"
    }

print("\n=== Self-BLEU Diversity Report ===")
for sentiment, stats in diversity_stats.items():
    print(f"{sentiment:10s}: {stats['count']:3d} reviews, Self-BLEU={stats['self_bleu']:.4f}, Quality={stats['quality']}")

#  2d. Sentiment consistency check 
# Use baseline model to predict sentiment of each generated review
# Flag mismatches, save llm_generated_flagged.csv

print("\n2d. Running sentiment consistency check...")
X_gen = tfidf.transform(gen_df["review"])
gen_df["lr_predicted_label"] = lr_model.predict(X_gen)

gen_df["mismatch"] = gen_df["label"] != gen_df["lr_predicted_label"]

flagged_df = gen_df[gen_df["mismatch"]]


flagged_df.to_csv("llm_generated_flagged.csv", index=False)
consistency_accuracy = (1 - len(flagged_df)/len(gen_df))*100 if len(gen_df) > 0 else 0
print(f"  Flagged {len(flagged_df)} mismatches")
print(f"  Sentiment consistency accuracy: {consistency_accuracy:.2f}%")

#  2e. Save outputs 
# Save llm_generated_300.csv and diversity_report.txt

print("\n2e. Saving outputs...")
gen_df[["review", "label", "movie"]].to_csv(
    "llm_generated_300.csv", index=False
)


diversity_report = "=== LLM Generated Reviews Diversity Report ===\n\n"
diversity_report += f"Total reviews generated: {len(gen_df)}\n"
diversity_report += f"Reviews after deduplication & filtering: {len(gen_df)}\n"
diversity_report += f"Generator: {'OpenRouter API (Mistral-7B)' if USE_API else 'Local Synthetic Generator'}\n\n"

diversity_report += "Self-BLEU Metrics (Per Sentiment Class):\n"
diversity_report += "-" * 60 + "\n"
for sentiment, stats in diversity_stats.items():
    diversity_report += f"\nSentiment: {sentiment}\n"
    diversity_report += f"  Count:              {stats['count']}\n"
    diversity_report += f"  Self-BLEU Score:   {stats['self_bleu']:.4f}\n"
    diversity_report += f"  Quality Flag:      {stats['quality']}\n"
    diversity_report += f"  Threshold:         {SELF_BLEU_THRESHOLD:.2f}\n"

overall_self_bleu = np.mean([stats['self_bleu'] for stats in diversity_stats.values()])
diversity_report += f"\nOverall Average Self-BLEU: {overall_self_bleu:.4f}\n"

diversity_report += f"\nSentiment Consistency Check:\n"
diversity_report += "-" * 60 + "\n"
diversity_report += f"Total mismatches: {len(flagged_df)}\n"
diversity_report += f"Accuracy:         {consistency_accuracy:.2f}%\n"

with open("diversity_report.txt", "w", encoding="utf-8") as f:
    f.write(diversity_report)

print("\n✓ Task 2 Complete!")
print("  - llm_generated_300.csv (" + str(len(gen_df)) + " reviews)")
print("  - llm_generated_flagged.csv (" + str(len(flagged_df)) + " flagged)")
print("  - prompt_template.txt")
print("  - diversity_report.txt")


✓ Using OpenRouter API with Mistral-7B (free)
Few-shot prompt designed and saved to prompt_template.txt

2b. Generating reviews in batches...

Generating 150 Positive reviews:
  Batch 1/8 (20 reviews)... API failed, using local generator
✓ Got 20 reviews
  Batch 2/8 (20 reviews)... ✓ Got 20 reviews
  Batch 3/8 (20 reviews)... ✓ Got 20 reviews
  Batch 4/8 (20 reviews)... ✓ Got 20 reviews
  Batch 5/8 (20 reviews)... ✓ Got 20 reviews
  Batch 6/8 (20 reviews)... ✓ Got 20 reviews
  Batch 7/8 (20 reviews)... ✓ Got 20 reviews
  Batch 8/8 (10 reviews)... ✓ Got 10 reviews
  Total Positive collected: 150

Generating 100 Negative reviews:
  Batch 1/5 (20 reviews)... API failed, using local generator
✓ Got 20 reviews
  Batch 2/5 (20 reviews)... ✓ Got 20 reviews
  Batch 3/5 (20 reviews)... ✓ Got 20 reviews
  Batch 4/5 (20 reviews)... ✓ Got 20 reviews
  Batch 5/5 (20 reviews)... ✓ Got 20 reviews
  Total Negative collected: 100

Generating 50 Neutral reviews:
  Batch 1/3 (20 reviews)... API failed, u

## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold â‰¥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [21]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pandas as pd
import time
import random
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

consolidated_base = pd.read_csv("consolidated_base.csv")
consolidated_base['label'] = consolidated_base['label'].str.strip().str.capitalize()

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

gold = pd.read_csv("gold_standard_100.csv")
gold['label'] = gold['label'].str.strip().str.capitalize()

baseline_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, C=1.0, random_state=42))
])
baseline_model.fit(gold['review'], gold['label'])

#  3a. Strategic sampling 
# From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)

def sample_by_class(df, label, n):
    subset = df[df['label'] == label].copy()
    subset = subset.assign(length=subset['review'].str.len())
    subset = subset.sort_values('length')
    return subset.head(n)[['review', 'label']]

pos_samples = sample_by_class(consolidated_base, 'Positive', 40)
neg_samples = sample_by_class(consolidated_base, 'Negative', 40)
neu_samples = sample_by_class(consolidated_base, 'Neutral', 20)

sampled_100 = pd.concat([pos_samples, neg_samples, neu_samples], ignore_index=True)

print(f"sampled {len(sampled_100)} reviews -> "
      f"positive: {len(pos_samples)} negative: {len(neg_samples)} neutral: {len(neu_samples)}")


#  3b. Translation pipeline 
# Translate each review English → Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits

hindi_translations = []

for i, review in enumerate(sampled_100['review']):
    try:
        hindi = GoogleTranslator(source='en', target='hi').translate(review)
        hindi_translations.append(hindi if hindi else review)
    except Exception as e:
        print(f"[row{i}] translation failed: {e}")
        hindi_translations.append(review)
        time.sleep(0.5)

sampled_100['hindi'] = hindi_translations


#  3c. Back-translation & BLEU 
# Translate Hindi → English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"

smoother = SmoothingFunction().method1

back_translations = []
bleu_scores = []
quality_flags = []

for i, row in sampled_100.iterrows():
    try:
        back_en = GoogleTranslator(source='hi', target='en').translate(row['hindi'])
        back_en = back_en if back_en else row['review']
    except Exception as e:
        print(f"[row{i}] back-translation failed: {e}")
        back_en = row['review']
    time.sleep(0.5)

    ref_tokens = nltk.word_tokenize(row['review'].lower())
    hyp_tokens = nltk.word_tokenize(back_en.lower())
    bleu = sentence_bleu([ref_tokens], hyp_tokens, weights=(0.5, 0.5), smoothing_function=smoother)
    bleu = round(float(bleu), 4)

    back_translations.append(back_en)
    bleu_scores.append(bleu)
    quality_flags.append("PASS" if bleu >= 0.3 else "FAIL")

sampled_100['back_translated'] = back_translations
sampled_100['bleu_score'] = bleu_scores
sampled_100['quality_flag'] = quality_flags

print(f"\nBLEU quality -> PASS: {quality_flags.count('PASS')}, "
      f"FAIL: {quality_flags.count('FAIL')}, "
      f"(avg BLEU: {sum(bleu_scores)/len(bleu_scores):.4f})")


#  3d. Sentiment preservation check 
# Predict sentiment on back-translated text, compare with original label

pred_back = baseline_model.predict(sampled_100['back_translated'].tolist())
sampled_100['predicted_sentiment'] = pred_back
sampled_100['sentiment_preserved'] = (
    sampled_100['label'].str.lower() == sampled_100['predicted_sentiment'].str.lower()
)

preserved_count = sampled_100['sentiment_preserved'].sum()
print(f"Sentiment preserved: {preserved_count}/{len(sampled_100)} "
      f"({100*preserved_count/len(sampled_100):.1f}%)")


#  3e. Manual verification 
# Print 5 random samples for inspection

random.seed(42)
five_indices = random.sample(range(len(sampled_100)), 5)

print("\n--- Manual Verification (5 random samples) ---")

for rank, idx in enumerate(five_indices, 1):
    row = sampled_100.iloc[idx]
    print(f"\nSample {rank}:")
    print(f" Original [{row['label']}]: {row['review']}")
    print(f" Hindi:                     {row['hindi']}")
    print(f" Back-EN:                   {row['back_translated']}")
    print(f" BLEU: {row['bleu_score']:.4f} | Quality: {row['quality_flag']} | "
          f"Sentiment preserved: {row['sentiment_preserved']}")


#  3f. Save 
# Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag

output_cols = ['review', 'label', 'hindi', 'back_translated', 'bleu_score', 'quality_flag']
sampled_100[output_cols].to_csv("bilingual_reviews.csv", index=False)

print(f"\nSaved bilingual_reviews.csv ({len(sampled_100)} rows)")
print(f"Columns: {output_cols}")


sampled 100 reviews -> positive: 40 negative: 40 neutral: 20

BLEU quality -> PASS: 99, FAIL: 1, (avg BLEU: 0.7544)
Sentiment preserved: 81/100 (81.0%)

--- Manual Verification (5 random samples) ---

Sample 1:
 Original [Neutral]: I have mixed feelings about this one. 6/10 at best.
 Hindi:                     इसके बारे में मेरी मिश्रित भावनाएँ हैं। सर्वोत्तम रूप से 6/10.
 Back-EN:                   I have mixed feelings about it. 6/10 at best.
 BLEU: 0.7787 | Quality: PASS | Sentiment preserved: True

Sample 2:
 Original [Positive]: I was hooked from the very first minute. I can't wait to see it again.
 Hindi:                     मैं पहले मिनट से ही आदी हो गया था। मैं इसे दोबारा देखने के लिए इंतजार नहीं कर सकता.
 Back-EN:                   I was hooked from the first minute. I can't wait to see it again.
 BLEU: 0.9129 | Quality: PASS | Sentiment preserved: True

Sample 3:
 Original [Positive]: The dialogue was absolutely phenomenal. Wow, just wow.
 Hindi:                     संवाद बिल

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [22]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class)
# TODO: Sample from consolidated_base, mix short and long reviews.
sampled_parts = []
per_class_target = 10

rng = np.random.default_rng()
sentiment_order = list(consolidated['label'].dropna().unique())
rng.shuffle(sentiment_order)

for sentiment in sentiment_order:
    class_samples = consolidated[consolidated['label'] == sentiment].copy()
    class_samples = class_samples.dropna(subset=['review']).drop_duplicates(subset=['review'])
    class_samples['review_len'] = class_samples['review'].str.len()

    short_pool = class_samples[class_samples['review_len'] <= 100]
    long_pool = class_samples[class_samples['review_len'] > 100]

    short_n = min(5, len(short_pool))
    long_n = min(5, len(long_pool))

    short_samples = short_pool.sample(short_n) if short_n else short_pool.head(0)
    long_samples = long_pool.sample(long_n) if long_n else long_pool.head(0)

    selected = pd.concat([short_samples, long_samples])

    if len(selected) < per_class_target:
        remaining = class_samples.drop(index=selected.index, errors='ignore')
        fill_n = min(per_class_target - len(selected), len(remaining))
        if fill_n:
            fill_samples = remaining.sample(fill_n)
            selected = pd.concat([selected, fill_samples])

    selected = selected.sample(frac=1).head(per_class_target) if len(selected) else selected
    print(f"Selected {len(selected)} samples for sentiment {sentiment}")
    sampled_parts.append(selected[['review', 'label']])

sampled = pd.concat(sampled_parts, ignore_index=True)
sampled = sampled.drop_duplicates(subset=['review']).reset_index(drop=True)

if len(sampled) > 30:
    sampled = sampled.groupby('label', group_keys=False).apply(lambda x: x.sample(min(10, len(x)))).reset_index(drop=True)

os.makedirs("audio_samples", exist_ok=True)
sampled.to_csv("audio_samples/sampled_reviews.csv", index=False)


#  4b. TTS generation
# TODO: For each review, generate audio with gTTS (tld="com").
# TODO: Save as mp3, then load with librosa and re-save as .wav via soundfile.

for directory in ["mp3", "wav"]:
    dir_path = os.path.join("audio_samples", directory)
    if os.path.exists(dir_path):
        for file in os.listdir(dir_path):
            file_path = os.path.join(dir_path, file)
            if os.path.isfile(file_path):
                os.remove(file_path)
    else:
        os.makedirs(dir_path, exist_ok=True)

for idx, row in sampled.iterrows():
    review_text = str(row['review'])
    sentiment = str(row['label'])
    mp3filename_base = f"audio_samples/mp3/review_{idx}_{sentiment}"
    wavfilename_base = f"audio_samples/wav/review_{idx}_{sentiment}"

    tts = gTTS(text=review_text, lang='en', tld='com')
    mp3_path = f"{mp3filename_base}.mp3"
    wav_path = f"{wavfilename_base}.wav"

    tts.save(mp3_path)
    audio, sr = librosa.load(mp3_path, sr=16000)
    sf.write(wav_path, audio, sr)

print("Audio generation complete:", len(sampled), "samples")

Selected 10 samples for sentiment Negative
Selected 10 samples for sentiment Neutral
Selected 10 samples for sentiment Positive
Audio generation complete: 30 samples


In [23]:

#  4c. Audio feature extraction 
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv

audio_rows = []

for wav_file in sorted(os.listdir("audio_samples/wav")):
    if not wav_file.lower().endswith(".wav"):
        continue
    
    wav_path = os.path.join("audio_samples/wav", wav_file)
    filename = os.path.basename(wav_path)

    stem = os.path.splitext(filename)[0]
    parts = stem.split("_", 2)
    sentiment = parts[2] if len(parts) == 3 else "Unknown"

    try:
        audio, sr = librosa.load(wav_path, sr=16000)
        duration = librosa.get_duration(y=audio, sr=sr)
        spectral_centroid = float(np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr)))
        zero_crossing_rate = float(np.mean(librosa.feature.zero_crossing_rate(y=audio)))
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        mfcc_mean = float(np.mean(mfccs))

        audio_rows.append({
            "filename": filename,
            "sentiment": sentiment,
            "duration": duration,
            "spectral_centroid": spectral_centroid,
            "zero_crossing_rate": zero_crossing_rate,
            "mfcc_mean": mfcc_mean,
        })
    except Exception as e:
        print(f"Skipping {filename}: {e}")

audio_data = pd.DataFrame(audio_rows, columns=[
    "filename", "sentiment", "duration", "spectral_centroid", "zero_crossing_rate", "mfcc_mean"
])
audio_data.to_csv("audio_features.csv", index=False)
print("audio_features.csv rows:", len(audio_data))
audio_data.head()

audio_features.csv rows: 30


,filename,sentiment,duration,spectral_centroid,zero_crossing_rate,mfcc_mean
0,review_0_Negative.wav,Negative,7.680,1974.300308,0.165519,-22.151901
1,review_10_Neutral.wav,Neutral,6.624,1787.988624,0.114063,-23.557400
2,review_11_Neutral.wav,Neutral,10.704,1751.626253,0.141001,-21.883709
3,review_12_Neutral.wav,Neutral,5.472,1956.759142,0.152196,-22.198624
4,review_13_Neutral.wav,Neutral,4.440,1860.035541,0.131920,-25.046574


In [24]:
#  4d. Whisper round-trip validation
# TODO: Transcribe each wav with Whisper.
# TODO: Compute WER (word-level Levenshtein distance / reference word count).
# TODO: Flag samples with WER > 0.25 and save audio_validation.csv.
import whisper

_whisper_model = whisper.load_model("tiny")

def _word_levenshtein_distance(ref_words, hyp_words):
    """Compute Levenshtein edit distance between two word lists."""
    m, n = len(ref_words), len(hyp_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if ref_words[i - 1] == hyp_words[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost,
            )
    return dp[m][n]

def compute_wer(reference, hypothesis):
    ref_words = re.findall(r"\w+", str(reference).lower())
    hyp_words = re.findall(r"\w+", str(hypothesis).lower())
    if len(ref_words) == 0:
        return 0.0 if len(hyp_words) == 0 else 1.0
    dist = _word_levenshtein_distance(ref_words, hyp_words)
    return dist / len(ref_words)

if 'sampled' in globals() and {'review', 'label'}.issubset(sampled.columns):
    sampled_for_lookup = sampled[['review', 'label']].reset_index(drop=True)
elif os.path.exists("audio_samples/sampled_reviews.csv"):
    sampled_for_lookup = pd.read_csv("audio_samples/sampled_reviews.csv")[['review', 'label']].reset_index(drop=True)
else:
    raise FileNotFoundError(
        "Missing sampled metadata. Run Task 4 cell 12 first to generate audio_samples/sampled_reviews.csv."
    )

sampled_lookup = sampled_for_lookup.to_dict("index")
validation_rows = []

for wav_file in sorted(os.listdir("audio_samples/wav")):
    if not wav_file.lower().endswith(".wav"):
        continue

    wav_path = os.path.join("audio_samples/wav", wav_file)

    base = os.path.splitext(wav_file)[0]
    parts = base.split("_", 2)
    if len(parts) < 3 or parts[0] != "review":
        continue

    try:
        sample_idx = int(parts[1])
    except ValueError:
        continue

    if sample_idx not in sampled_lookup:
        continue

    reference_text = sampled_lookup[sample_idx]["review"]
    true_label = sampled_lookup[sample_idx]["label"]

    try:
        audio_wave, sr = librosa.load(wav_path, sr=16000)
        result = _whisper_model.transcribe(audio_wave, fp16=False)
        transcript = result.get("text", "").strip()
        wer = compute_wer(reference_text, transcript)
        quality_flag = "FAIL" if wer > 0.25 else "PASS"
    except Exception as e:
        transcript = ""
        wer = np.nan
        quality_flag = f"ERROR: {type(e).__name__}"
        print(f"Transcription failed for {wav_file}: {e}")

    validation_rows.append({
        "filename": wav_file,
        "label": true_label,
        "reference": reference_text,
        "transcript": transcript,
        "wer": round(wer, 4),
        "quality_flag": quality_flag,
    })

audio_validation = pd.DataFrame(validation_rows)
audio_validation.to_csv("audio_validation.csv", index=False)

print("Whisper validation complete.")
print("Total files processed:", len(audio_validation))
if len(audio_validation) and "wer" in audio_validation.columns:
    print("High-WER files (>0.25):", int((audio_validation["wer"] > 0.25).sum()))
audio_validation.head()

Whisper validation complete.
Total files processed: 30
High-WER files (>0.25): 1


,filename,label,reference,transcript,wer,quality_flag
0,review_0_Negative.wav,Negative,"The visuals were stellar, but the story was wo...","The visuals were stellar, but the story was wo...",0.0000,PASS
1,review_10_Neutral.wav,Neutral,"Ideally, this should have been great, but some...","Ideally, this should have been great, but some...",0.0000,PASS
2,review_11_Neutral.wav,Neutral,I walked in with low expectations and they wer...,I walked in with low expectations and they wer...,0.0000,PASS
3,review_12_Neutral.wav,Neutral,"The soundtrack was decent, but nothing special...","The soundtrack was decent, but nothing special...",0.0000,PASS
4,review_13_Neutral.wav,Neutral,"It was... fine. If you're bored, give it a shot.",It was fine. If your board give it a shot.,0.2727,FAIL


## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [25]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset 
# Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv

print("=== Task 5: Final Dataset Assembly & Model Evaluation ===\n")

# Load all datasets
print("Loading datasets...")
try:
    consolidated_base = pd.read_csv("consolidated_base.csv")
    print(f"  consolidated_base.csv: {len(consolidated_base)} rows")
except FileNotFoundError:
    raise FileNotFoundError("consolidated_base.csv not found. Run Task 1 first.")

try:
    augmented_classical = pd.read_csv("augmented_classical.csv")
    print(f"  augmented_classical.csv: {len(augmented_classical)} rows")
except FileNotFoundError:
    raise FileNotFoundError("augmented_classical.csv not found. Run Task 1 first.")

try:
    llm_generated = pd.read_csv("llm_generated_300.csv")
    print(f"  llm_generated_300.csv: {len(llm_generated)} rows")
except FileNotFoundError:
    raise FileNotFoundError("llm_generated_300.csv not found. Run Task 2 first.")

try:
    llm_flagged = pd.read_csv("llm_generated_flagged.csv")
    print(f"  llm_generated_flagged.csv: {len(llm_flagged)} rows (to exclude)")
except FileNotFoundError:
    llm_flagged = pd.DataFrame()  # Empty if no flagged file
    print(f"  llm_generated_flagged.csv: not found (proceeding with no exclusions)")

try:
    bilingual_reviews = pd.read_csv("bilingual_reviews.csv")
    print(f"  bilingual_reviews.csv: {len(bilingual_reviews)} rows")
except FileNotFoundError:
    raise FileNotFoundError("bilingual_reviews.csv not found. Run Task 3 first.")

# Exclude flagged LLM reviews
if len(llm_flagged) > 0:
    flagged_reviews_set = set(llm_flagged["review"].tolist())
    llm_generated_clean = llm_generated[~llm_generated["review"].isin(flagged_reviews_set)].copy()
    print(f"\nExcluded {len(llm_flagged)} flagged LLM reviews")
    print(f"  llm_generated_300.csv after exclusion: {len(llm_generated_clean)} rows")
else:
    llm_generated_clean = llm_generated.copy()

# Prepare datasets: select only review and label columns
print("\nPreparing datasets for merging...")

# consolidated_base: keep review, label
consolidated_base_prep = consolidated_base[["review", "label"]].copy()
consolidated_base_prep["source"] = "consolidated_base"

# augmented_classical: keep review, label
augmented_classical_prep = augmented_classical[["review", "label"]].copy()
augmented_classical_prep["source"] = "augmented_classical"

# llm_generated: keep review, label
llm_generated_prep = llm_generated_clean[["review", "label"]].copy()
llm_generated_prep["source"] = "llm_generated"

# bilingual_reviews: use review, label
bilingual_prep = bilingual_reviews[["review", "label"]].copy()
bilingual_prep["source"] = "bilingual"

# Validate label consistency
print("\nValidating label values...")
all_dfs = [consolidated_base_prep, augmented_classical_prep, llm_generated_prep, bilingual_prep]
all_labels = set()
for df in all_dfs:
    all_labels.update(df["label"].unique())
print(f"  Unique labels across all datasets: {sorted(all_labels)}")

# Merge all datasets
print("\nMerging all datasets...")
final_dataset = pd.concat(
    [consolidated_base_prep, augmented_classical_prep, llm_generated_prep, bilingual_prep],
    ignore_index=True
)
print(f"  Combined size: {len(final_dataset)} rows")

# Deduplicate by review text
print("\nDeduplicating by review text...")
final_dataset = final_dataset.drop_duplicates(subset="review", keep="first").reset_index(drop=True)
print(f"  After deduplication: {len(final_dataset)} unique reviews")

# Show class distribution
print("\nFinal class distribution:")
class_dist = final_dataset["label"].value_counts()
for label, count in class_dist.items():
    print(f"  {label}: {count} ({100*count/len(final_dataset):.1f}%)")

# Save final augmented dataset
final_dataset[["review", "label"]].to_csv("final_augmented_dataset.csv", index=False)
print(f"\nSaved: final_augmented_dataset.csv ({len(final_dataset)} rows)")

#  5b. Black-Box Evaluation 
# Compare baseline vs augmented accuracy

print("\n" + "="*60)
print("BLACK-BOX EVALUATION")
print("="*60 + "\n")

# Initialize evaluator
evaluator = BlackBoxEvaluator()

# Load test set
test_df = pd.read_csv("gold_standard_100.csv")
print(f"Test set: gold_standard_100.csv ({len(test_df)} samples)\n")

# Baseline evaluation (consolidated_base only)
baseline_acc = evaluator.run_evaluation(
    consolidated_base,
    test_df,
    model_name="Baseline (consolidated_base)"
)

# Augmented evaluation (final_augmented_dataset)
augmented_acc = evaluator.run_evaluation(
    final_dataset,
    test_df,
    model_name="Augmented (final_augmented_dataset)"
)

# Print comparison
print("="*60)
print("FINAL COMPARISON")
print("="*60)
print(f"Baseline accuracy:  {baseline_acc*100:6.2f}%")
print(f"Augmented accuracy: {augmented_acc*100:6.2f}%")
improvement = augmented_acc - baseline_acc
print(f"Improvement:        {improvement:+6.2f}% ({improvement*100:+.2f} pp)")
print("="*60 + "\n")

if improvement > 0:
    print(f"✓ SUCCESS: Augmentation improved accuracy by {improvement*100:.2f} percentage points!")
elif improvement == 0:
    print("Note: Accuracies are equal.")
else:
    print(f"Note: Augmentation decreased accuracy by {abs(improvement)*100:.2f} percentage points.")
    print("This can occur due to the quality/diversity tradeoff in synthetic data generation.")


=== Task 5: Final Dataset Assembly & Model Evaluation ===

Loading datasets...
  consolidated_base.csv: 328 rows
  augmented_classical.csv: 449 rows
  llm_generated_300.csv: 221 rows
  llm_generated_flagged.csv: 148 rows (to exclude)
  bilingual_reviews.csv: 100 rows

Excluded 148 flagged LLM reviews
  llm_generated_300.csv after exclusion: 73 rows

Preparing datasets for merging...

Validating label values...
  Unique labels across all datasets: ['Negative', 'Neutral', 'Positive']

Merging all datasets...
  Combined size: 950 rows

Deduplicating by review text...
  After deduplication: 522 unique reviews

Final class distribution:
  Positive: 204 (39.1%)
  Negative: 160 (30.7%)
  Neutral: 158 (30.3%)

Saved: final_augmented_dataset.csv (522 rows)

BLACK-BOX EVALUATION

Initializing Black-Box Embedder...


ValueError: The provided filename text_embedder.pt does not exist